In [5]:
import requests
from collections import Counter

# --------------------------------------------------
# 1) FETCH UNIPROT SEQUENCE
# --------------------------------------------------
url = "https://rest.uniprot.org/uniprotkb/P04637.fasta"
r = requests.get(url)
r.raise_for_status()

fasta = r.text
full_seq = "".join(line.strip() for line in fasta.splitlines() if not line.startswith(">"))

# choose short subsequence
start = 0
length = 5
seq = full_seq[start:start+length]

print("Subsequence:", seq)


# --------------------------------------------------
# 2) AMINO ACID MASS TABLE
# --------------------------------------------------
AA_MASS = {
    'G':57,'A':71,'S':87,'P':97,'V':99,'T':101,'C':103,'I':113,'L':113,
    'N':114,'D':115,'K':128,'Q':128,'E':129,'M':131,'H':137,'F':147,
    'R':156,'Y':163,'W':186
}
MASSES = list(set(AA_MASS.values()))

true_peptide = [AA_MASS[a] for a in seq]


# --------------------------------------------------
# 3) CYCLIC SPECTRUM
# --------------------------------------------------
def cyclic_spectrum(peptide):
    n = len(peptide)
    total = sum(peptide)
    spec = [0]
    doubled = peptide + peptide
    for L in range(1, n):
        for i in range(n):
            spec.append(sum(doubled[i:i+L]))
    spec.append(total)
    return sorted(spec)

spectrum = cyclic_spectrum(true_peptide)
parent_mass = max(spectrum)


# --------------------------------------------------
# 4) BASIC FUNCTIONS
# --------------------------------------------------
def mass(peptide):
    return sum(peptide)

def expand(peptides):
    return [p+[m] for p in peptides for m in MASSES]

def linear_spectrum(peptide):
    spec=[0]
    for i in range(len(peptide)):
        s=0
        for j in range(i,len(peptide)):
            s+=peptide[j]
            spec.append(s)
    return sorted(spec)

def score_linear(peptide, spectrum):
    theo = Counter(linear_spectrum(peptide))
    exp  = Counter(spectrum)
    return sum(min(theo[m], exp[m]) for m in theo)

def score_cyclic(peptide, spectrum):
    theo = Counter(cyclic_spectrum(peptide))
    exp  = Counter(spectrum)
    return sum(min(theo[m], exp[m]) for m in theo)


# --------------------------------------------------
# 5) TRIM WITH DISPLAY
# --------------------------------------------------
def trim(leaderboard, spectrum, N):

    scored = [(p, score_linear(p, spectrum)) for p in leaderboard]
    scored.sort(key=lambda x:x[1], reverse=True)

    print("\n--- Sorted leaderboard ---")
    for p,s in scored:
        print(p, " mass=", mass(p), " linear score=", s)

    if len(scored) <= N:
        return [p for p,_ in scored]

    cutoff = scored[N-1][1]
    print("Cutoff score:", cutoff)

    trimmed = [p for p,s in scored if s >= cutoff]

    print("\n--- After trimming ---")
    for p in trimmed:
        print(p, " mass=", mass(p), " linear score=", score_linear(p,spectrum))

    return trimmed


# --------------------------------------------------
# 6) LEADERBOARD ALGORITHM (FULL TRACE)
# --------------------------------------------------
def leaderboard_cyclopeptide_sequencing(spectrum, N=10):

    leaderboard=[[]]
    leader=[]
    step=1

    while leaderboard:

        print("\n==============================")
        print("STEP", step)
        print("==============================")

        leaderboard = expand(leaderboard)
        print("Expanded candidates:", len(leaderboard))

        new=[]
        for pep in leaderboard:
            m = mass(pep)

            # compare FULL peptides using CYCLIC score (correct fix)
            if m == parent_mass:
                if score_cyclic(pep, spectrum) > score_cyclic(leader, spectrum):
                    leader = pep

            if m <= parent_mass:
                new.append(pep)

        leaderboard = trim(new, spectrum, N)
        step += 1

    return leader


# --------------------------------------------------
# 7) RUN
# --------------------------------------------------
result = leaderboard_cyclopeptide_sequencing(spectrum, N=5)

print("\nFINAL LEADER PEPTIDE:", result)
print("TRUE PEPTIDE:", true_peptide)


Subsequence: MEEPQ

STEP 1
Expanded candidates: 18

--- Sorted leaderboard ---
[128]  mass= 128  linear score= 2
[97]  mass= 97  linear score= 2
[129]  mass= 129  linear score= 2
[131]  mass= 131  linear score= 2
[99]  mass= 99  linear score= 1
[101]  mass= 101  linear score= 1
[163]  mass= 163  linear score= 1
[71]  mass= 71  linear score= 1
[103]  mass= 103  linear score= 1
[137]  mass= 137  linear score= 1
[113]  mass= 113  linear score= 1
[114]  mass= 114  linear score= 1
[115]  mass= 115  linear score= 1
[147]  mass= 147  linear score= 1
[87]  mass= 87  linear score= 1
[57]  mass= 57  linear score= 1
[186]  mass= 186  linear score= 1
[156]  mass= 156  linear score= 1
Cutoff score: 1

--- After trimming ---
[128]  mass= 128  linear score= 2
[97]  mass= 97  linear score= 2
[129]  mass= 129  linear score= 2
[131]  mass= 131  linear score= 2
[99]  mass= 99  linear score= 1
[101]  mass= 101  linear score= 1
[163]  mass= 163  linear score= 1
[71]  mass= 71  linear score= 1
[103]  mass= 